<a href="https://colab.research.google.com/github/gibsonx/jlpt_simulator/blob/dev/graphs/n3/outliner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
if 'google.colab' in str(get_ipython()):
    !git clone https://github.com/gibsonx/jlpt_simulator.git
    %cd jlpt_simulator
    !git checkout dev
    !apt-get install python3-dev graphviz libgraphviz-dev pkg-config
    !pip install -r requirements.txt
else:
  print('Not running on CoLab')


Not running on CoLab


In [2]:
import json
import logging
import random
import time
import pandas as pd
import yaml
import inspect
from tqdm import tqdm
import os
from libs.Logger import logger
from datetime import datetime
from docx import Document
from html4docx import HtmlToDocx
import uuid
from libs.CosmosMongoDB import CosmosMongoDB
from libs.LLMs import *
from IPython.display import display, Markdown, HTML
import datetime
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
from libs.Utils import render_to_html,collect_vocabulary,_load_vocab_and_resources
from graphs.common.Schema import Outline
from langchain_core.prompts import ChatPromptTemplate
from graphs.common.ExamGenerator import ExamGenerator
load_dotenv()

True

## Define Exam ID, Topics and Vocabulary

In [3]:
exam_uid = str(uuid.uuid1())
level = 'n1'
vocab, topics, grammar = _load_vocab_and_resources(level)

#### load Models

In [4]:
instruction_backup = """
Section 1: vocabulary
- 問題1 のことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい (kanji_reading) 8 questions in total: 3 are nouns, 3 are verbs, 1 is an adjective, and 1 is an adverb
- 問題2 このことばを漢字で書くとき、最もよいものを、1・2・3・4から一つえらびなさい (write_kanji) 6 questions in total: 2 nouns, 2 verbs, 1 adjective, and 1 adverb.
- 問題3（　）に入れるのに最もよいものを、1・2・3・4から一つえらびなさい。 (word_meaning) 11 questions in total: 4 are nouns, 4 are verbs, 2 are adjectives, and 1 is an adverb.
- 問題4 に意味が最も近いものを、1・2・3・4から一つえらびなさい。(synonym_substitution) 5 questions in total: 2 are nouns, 2 are verbs, and 1 is an adjective.
- 問題5 つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。 (word_usage) 5 questions in total: 3 noun, and 2 verbs.

Section 2: Grammar
- 問題6 つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。(sentence_grammar) 13 questions in total: the first 1 is honorific speech, next 1 adverb, 1 auxiliary word, and other 10 different sentence structures
- 問題7 つぎの文の ★ に入る最もよいものを、1・2・3・4から一つえらびなさい。(sentence_sort) 5 questions in total.
- 問題8 つぎの文章を読んで、文章全体の内容を考えて、文中の 19 から 22 の中に入る最もよいものを、1・2・3・4から一つえらびなさい (sentence_structure) 1 question

Section 3: Reading Comprehension
- 問題1-1 つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい (short_passage_mail_read): 1 article
- 問題1-2 つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい (short_passage_notification_read): 1 article
- 問題1-3 つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい (short_passage_narrative_read): 2 articles
- 問題2 つぎの(1)と(2)の文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。 (midsize_passage_read): 2 articles
- 問題3 つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。(long_passage_read): 1 article
- 問題4 これを読んで、下の質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい Information retrieval (info_retrieval): 1 article

Section 4: Listening Comprehension
- 問題1 では、まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。 (topic_understanding): 6 question
- 問題2 では、まず質問を聞いてください。そのあと、問題用紙を見てください。読む時間があります。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。
 (keypoint_understanding): 6 question
- 問題3では、問題用紙（もんだいようし）に何（なに）も いんさつされていません。この問題（もんだい）は、ぜんたいとして どんな ないようかを聞（き）く 問題（もんだい）です。話（はなし）の前（まえ）に 質問（しつもん）は ありません。まず 話（はなし）を 聞（き）いてください。それから、質問（しつもん）と せんたくし を聞（き）いて、1から4の中（なか）から、最（もっと）も よい ものを 一（ひと）つ えらんでください。(summary_understanding) 3 question
- 問題4 では、元を見ながら質問を聞いてください。やじるし（➔）の人は何と言いますか。1 から 3 の中から、最もよいものを一つえらんでください。 (active_expression): 4 questions
- 問題5 では、問題用紙に何もいんさつしていません。まず文を聞いてください。それから、そのへんしを聞いて、1 から 3 の中から、最もよいものを一つえらんでください。 (immediate_ack): 9 questions
"""

instruction = """
Section 1: vocabulary
- 問題1 のことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい (kanji_reading) 8 questions in total: 3 are nouns, 3 are verbs, 1 is an adjective,  adjective.
- 問題5 つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。 (word_usage) 5 questions in total: 3 noun, and 2 verbs.

Section 2: Grammar
- 問題6 つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。(sentence_grammar) 13 questions in total: the first 1 is honorific

Section 3: Reading Comprehension
- 問題1-1 つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい (short_passage_mail_read): 1 article

Section 4: Listening Comprehension
- 問題1 では、まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。 (topic_understanding): 6 question
"""

direct_gen_outline_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a Japanese teacher tasked with creating an outline for a JLPT N3 level exam paper."
                "The overall difficulty should be appropriate for the N3 level.\n"
                "The exam paper should include a mix of moderately difficult and very difficult topics to accurately assess proficiency.\n\n"
                "Please ensure the following requirements are met:\n\n"
                "subsection_title in Subsection must be written in English.\n\n"
                "For Section 1 - vocabulary:\n"
                "- Select vocabulary words from the 'vocabulary' list, ensuring that 80% of the topics are very difficult.\n"
                "- The topic words in 問題1 (kanji_reading) and 問題5 (word_usage) must be written in Kanji while other topic words use Japanese kana.\n"
                "For Section 2 - Grammar:\n"
                "- Randomly select topics from 'TopicList' and grammars from 'GrammarList'.\n"
                "- For 問題8, include one question that integrates 4 different grammar points.\n"
                "- The grammar used must be appropriate and consistent with the chosen topic.\n\n"
                "For Section 3 - Reading Comprehension and Listening Comprehension:\n"
                "- Randomly choose topics from 'TopicList'.\n\n"
                "For Section 4: Listening Comprehension:\n"
                "- Randomly choose topics from 'TopicList'.\n\n"
                "Additionally:\n"
                "- Each topic word should be unique and must not be repeated in the outline.\n"
                "- Follow the provided exam instructions carefully to determine the number of questions and content for each section.\n"
                "- Finally, write the full outline of the examination paper in Japanese, including question topics as per the instructions.\n\n"
                f"Instruction: {instruction}"
            ),
        ),
        ("user", "TopicList: {topic_list}, vocabulary: {vocab_dict}, GrammarList: {grammar_list}"),
    ]
)


## Generate an Outliner of the Eaxm

In [5]:
generate_outline_direct = direct_gen_outline_prompt | azure_llm.with_structured_output(Outline)
initial_outline = generate_outline_direct.invoke({"topic_list": topics, "vocab_dict": vocab, "grammar_list": grammar })
logger.info("Outline of the exam: \n\n {}".format(initial_outline.as_str))

2025-09-17 11:50:26,660 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-09-17 11:50:26,737 - INFO - jlpt - Outline of the exam: 

 # 日本語能力試験 JLPT N3 模擬試験

## 第1部　語彙（Vocabulary）

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。全8問（名詞3、動詞3、形容詞2）。80%が難易度が非常に高い語彙。

- **ふんがい**
- **かきまわす**
- **しこう**
- **はげむ**
- **ともなう**
- **きゅうくつ**
- **ぶじょく**
- **しあがり**

### word_usage

問題5：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。全5問（名詞3、動詞2）。80%が難易度が非常に高い語彙。

- **みつど**
- **シナリオ**
- **きれめ**
- **のりこむ**
- **かきまわす**

## 第2部　文法（Grammar）

### sentence_grammar

問題6：つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。全13問。最初の1問は敬語（尊敬語）を使用。各問題はトピックと文法を組み合わせて出題。最後の1問は4つの文法を統合した難問。

- **レストランで食べ物を注文する**尊敬語（お～になる）
- **割引交渉**...てからでないと／...てからでなければ
- **交通状況について話す**...ほど...ない
- **天気の状況について話す**...そうだ（伝聞）
- **支払い方法について話す**...ようにする
- **領収書を求める**...ても／...でも
- **趣味について話す**...ことにしている
- **仕事のプロジ

In [6]:
display(Markdown(initial_outline.as_str))

# 日本語能力試験 JLPT N3 模擬試験

## 第1部　語彙（Vocabulary）

### kanji_reading

問題1：ことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。全8問（名詞3、動詞3、形容詞2）。80%が難易度が非常に高い語彙。

- **ふんがい**
- **かきまわす**
- **しこう**
- **はげむ**
- **ともなう**
- **きゅうくつ**
- **ぶじょく**
- **しあがり**

### word_usage

問題5：つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。全5問（名詞3、動詞2）。80%が難易度が非常に高い語彙。

- **みつど**
- **シナリオ**
- **きれめ**
- **のりこむ**
- **かきまわす**

## 第2部　文法（Grammar）

### sentence_grammar

問題6：つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。全13問。最初の1問は敬語（尊敬語）を使用。各問題はトピックと文法を組み合わせて出題。最後の1問は4つの文法を統合した難問。

- **レストランで食べ物を注文する**尊敬語（お～になる）
- **割引交渉**...てからでないと／...てからでなければ
- **交通状況について話す**...ほど...ない
- **天気の状況について話す**...そうだ（伝聞）
- **支払い方法について話す**...ようにする
- **領収書を求める**...ても／...でも
- **趣味について話す**...ことにしている
- **仕事のプロジェクトについて話す**...つもりだ
- **家族について話す**...ようだ
- **旅行の計画について話す**...たら...のに
- **最近の映画について話す**...らしい
- **健康とフィットネスについて話す**...べきだ
- **日本の祭りや文化イベントについて話す**...たばかり／...ようにする／...そうだ（伝聞）／...ことにしている

## 第3部　読解（Reading Comprehension）

### short_passage_mail_read

問題1-1：つぎの文章を読んで、質問に答えなさい。答えは、1・2・3・4から最もよいものを一つえらびなさい。トピックは『友人へのプレゼント選びについて話す』。

- **友人へのプレゼント選びについて話す**

## 第4部　聴解（Listening Comprehension）

### topic_understanding

問題1：まず質問を聞いてください。それから話を聞いて、問題用紙の1から4の中から、最もよいものを一つえらんでください。全6問。各問は異なるトピックを扱う。

- **店で価格を尋ねる**
- **バスの時刻表を尋ねる**
- **週末の予定について話す**
- **おすすめを尋ねる**
- **家事の分担について話す**
- **地元の観光名所について話す**

## Data Preparation

In [7]:
full_exam_generator = ExamGenerator(level="n1", exam_type="full_exam", db_collection="n1-full-exam")
inserted_id, outline = full_exam_generator._generate_and_store_paper(instruction=direct_gen_outline_prompt)

2025-09-17 11:50:41,259 - INFO - HTTP Request: POST https://ai-rolandaws880125ai409947751408.openai.azure.com/openai/deployments/gpt-4.1/chat/completions?api-version=2025-01-01-preview "HTTP/1.1 200 OK"
2025-09-17 11:50:41,262 - INFO - jlpt - Outline of the exam:

# 日本語能力試験 JLPT N3 模擬試験

## 第1部：語彙

### kanji_reading

問題1 のことばの読み方として最もよいものを、1・2・3・4から一つえらびなさい。全8問（名詞3、動詞3、形容詞2）。80%は難易度が非常に高い語彙を使用。

- **巧み**
- **施す**
- **補給**
- **討論**
- **褒める**
- **憧れ**
- **しなやか**
- **賢明**

### word_usage

問題5 つぎのことばの使い方として最もよいものを、1・2・3・4から一つえらびなさい。全5問（名詞3、動詞2）。80%は難易度が非常に高い語彙を使用。

- **策戦**
- **慎重**
- **領収書**
- **味わう**
- **指差す**

## 第2部：文法

### sentence_grammar

問題6 つぎの文の（　　　）に入れるのに最もよいものを、１・２・３・４から一つえらびなさい。全13問。最初の1問は敬語を使用。各文はトピックリストから選択したテーマに基づき、適切な文法を使用。

- **レストランで食べ物を注文する**...ていただけますか
- **交通手段について話す**...によって／...によっては
- **週末の予定について話す**...つもりだ
- **支払い方法について話す**...ことになっている
- **家族について話す**...ようだ
- **旅行の計画について話す**...予定だ
- **健康とフィットネスについて話す**...ようにしている
- **最近の映画について話す**...らしい
- **技術について話す**...によると
- **教育について

In [ ]:
db_client = CosmosMongoDB(
    os.environ['AZURE_MONGO_CONNECTION'],
    os.environ['AZURE_MONGO_DBNAME'],
    os.environ['AZURE_MONGO_COLLECTION']
)

# Insert the object
inserted_id = db_client.insert_one(outline)
print("Inserted document ID:", inserted_id)

In [ ]:
html_output = render_to_html(outline['sections'])
display(HTML(html_output))

In [ ]:
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
output_folder = "output"
html_filename = f"JLPT_{timestamp}.html"
html_filepath = os.path.join(os.getenv("PROJECT_PATH"),output_folder,html_filename)
with open(html_filepath, "w", encoding="utf-8") as file:
    file.write(html_output)

In [ ]:
def save_html_as_docx(filename,html, output_folder="output"):
    doc = Document()
    # Set 1" margins
    parser = HtmlToDocx()
    parser.add_html_to_document(html, doc)

    os.makedirs(output_folder, exist_ok=True)
    path = os.path.join(output_folder, filename)
    doc.save(path)
    print("Generated:", path)
    return path

In [ ]:
doc_filename = f"JLPT_{timestamp}.docx"
doc_filepath = os.path.join(output_folder,doc_filename)
save_html_as_docx(doc_filename, html_output,output_folder)

In [ ]:
# import paramiko
# 
# # Load the private key from the environment variable
# # Parse the private key
# key_file_path = 'temp_id_ed25519'
# pkey = paramiko.Ed25519Key.from_private_key_file(key_file_path)
# 
# # SSH and SFTP parameters
# hostname = "58.246.203.10"
# username = "hwu"
# port = 22
# 
# remote_html_file_path = f"/var/www/html/{html_filename}"
# remote_doc_file_path = f"/var/www/html/{doc_filename}"
# 
# # Establish SSH connection
# ssh = paramiko.SSHClient()
# ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
# ssh.connect(hostname=hostname, port=port, username=username, pkey=pkey)
# 
# # Transfer the file
# sftp = ssh.open_sftp()
# sftp.put(html_filepath, remote_html_file_path)
# print(f"File transferred to {hostname}:{remote_html_file_path}")
# sftp.put(doc_filepath, remote_doc_file_path)
# print(f"File transferred to {hostname}:{remote_doc_file_path}")
# 
# # Clean up
# sftp.close()
# ssh.close()